# Capstone Project: Dual-Stage Model Training & Evaluation
## Dual-Stage Network Intrusion Detection System (NIDS)
**Candidate:** Adarsh Kumar Sharma | **Roll No:** 2301420058 | **Program:** B.Tech CSE (Data Science)
**Objective:**
1. **Stage 1**: Train and compare high-speed Binary Classifiers (`Decision Tree`, `Logistic Regression`, `MLP Neural Network`).
2. **Stage 2**: Train Multi-Class Classifier (`Random Forest`) on attack flows to identify specific threat taxonomy.
3. Save models and preprocessors for deployment in the Streamlit Dashboard and REST API.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Add project root to sys.path
sys.path.insert(0, "..")
from src.config import load_config
from src.data_loader import load_dataset, prepare_labels
from src.preprocessor import FlowPreprocessor
from src.models import DualStageNIDS, create_classifier
from src.evaluate import compute_metrics, compute_per_attack_recall

cfg = load_config()
print("Modules imported successfully.")

### 1. Ingest Dataset & Feature Preprocessing
We load the flow data, stratify split (75% train / 25% test), and fit the `FlowPreprocessor` exclusively on training data to prevent data leakage.

In [ ]:
df = load_dataset(cfg["paths"]["primary_dataset"], max_rows=100000)
df = prepare_labels(df, cfg["attack_taxonomy"])

y_binary = df["is_attack"].values
y_multi = df["attack_category"].values
X_raw = df.drop(columns=["Label", "is_attack", "attack_category"])

X_train, X_test, y_bin_tr, y_bin_te, y_mul_tr, y_mul_te = train_test_split(
    X_raw, y_binary, y_multi, test_size=0.25, stratify=y_binary, random_state=42
)

preprocessor = FlowPreprocessor()
X_tr_trans = preprocessor.fit_transform(X_train)
X_te_trans = preprocessor.transform(X_test)

print(f"Train shape: {X_tr_trans.shape} | Test shape: {X_te_trans.shape}")

### 2. Stage 1 Binary Model Benchmark: Decision Tree vs Logistic Regression vs MLP

In [ ]:
candidates = {
    "Decision Tree": create_classifier("decision_tree"),
    "Logistic Regression": create_classifier("logistic_regression"),
    "Neural Net (MLP)": create_classifier("mlp")
}

benchmark_results = []
for name, clf in candidates.items():
    t0 = time.perf_counter()
    clf.fit(X_tr_trans, y_bin_tr)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    preds = clf.predict(X_te_trans)
    infer_time = (time.perf_counter() - t0) / len(X_te_trans) * 1e6  # ms per 1k flows

    probas = clf.predict_proba(X_te_trans)[:, 1]
    m = compute_metrics(y_bin_te, preds, probas)
    m["Model"] = name
    m["Train Time (s)"] = round(train_time, 2)
    m["Inference Latency (ms/1k)"] = round(infer_time, 2)
    benchmark_results.append(m)

df_res = pd.DataFrame(benchmark_results).set_index("Model")
df_res

### 3. Dual-Stage Architecture Training & Model Export

In [ ]:
dual_nids = DualStageNIDS(
    stage1_model_type="decision_tree",
    stage2_model_type="random_forest"
)
dual_nids.fit(X_tr_trans, y_bin_tr, y_mul_tr)

# Export artifacts
models_dir = cfg["paths"]["models_dir"]
os.makedirs(models_dir, exist_ok=True)
preprocessor.save(os.path.join(models_dir, "preprocessor.joblib"))
dual_nids.save(os.path.join(models_dir, "dual_stage_model.joblib"))

print("Model training complete and artifacts persisted!")

### 4. Per-Attack Category Recall Evaluation

In [ ]:
bin_preds, atk_probs, multi_preds = dual_nids.predict(X_te_trans)
per_attack = compute_per_attack_recall(y_mul_te, multi_preds)
per_attack